In [1]:
%matplotlib tk

# Eval data inspection (new layout)

This notebook replaces `Eval_data_inspection_stats` for the new artifacts written by `eval.py`.

Expected layout:
```
<eval_dir>/
  summary.json
  slim_traces.json
  trees_seed_<seed>.npz
```

Capabilities:
- load many eval dirs into a single table
- pick best checkpoint/run by different metrics
- plot trajectories (replacement for `plot_seeker_trajectory`)
- plot MCTS trees from packed `.npz` (replacement for `plot_mcts_tree_xy_limited`)
- plot step debug view (replacement for `plot_dbg_step`)


In [2]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

from plot_utils_eval import (
    load_eval_dir,
    load_packed_trees,
    slice_step_tree,
    plot_seeker_trajectory_slim,
    plot_mcts_tree_xy_limited_np,
    plot_eval_step,
)


## Point this notebook at your results directory

Set `RESULTS_ROOT` to the directory that contains your run folders (the ones that have `eval/<ckpt>` subfolders).


In [3]:
RESULTS_ROOT = Path("runs")  # <- change

# Optional: only include eval directories matching this regex (full path)
EVAL_DIR_NAME_RE = re.compile(r'.*')


## Discover eval directories

We look for folders that contain both a `summary.json` and a `slim_traces.json`.


In [4]:
def find_eval_dirs(results_root: Path):
    eval_dirs = []
    for p in results_root.rglob('summary.json'):
        eval_dir = p.parent
        if (eval_dir / 'slim_traces.json').exists():
            eval_dirs.append(eval_dir)
    eval_dirs = [d for d in eval_dirs if EVAL_DIR_NAME_RE.match(str(d))]
    return sorted(set(eval_dirs))

eval_dirs = find_eval_dirs(RESULTS_ROOT)
print(f"Found {len(eval_dirs)} eval dirs")
eval_dirs[:10]


Found 102 eval dirs


[WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000020000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000040000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000060000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000080000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000100000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000120000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000140000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000160000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000180000'),
 WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000200000')]

In [5]:
eval_dirs[0]

WindowsPath('runs/base__seed42__9567da53/2d/eval/ckpt_step_000020000')

In [6]:
eval_run = load_eval_dir(eval_dirs[0])

In [7]:
type(eval_run.slim_traces[0])

dict

In [8]:
plot_seeker_trajectory_slim(
    eval_run.slim_traces[0],
    title=f"test",
    L=10,
)

In [9]:
step = 0
slim = next(s for s in eval_run.slim_traces if int(s['seed']) == 1000)

npz_path = eval_run.trees_by_seed[1000]
if npz_path is None:
    raise FileNotFoundError(f"No trees file for seed={seed} in {er.eval_dir}")

packed = load_packed_trees(npz_path)
tree0 = slice_step_tree(packed, step)

chosen_idx = None
if 'chosen_idx' in slim and step < len(slim['chosen_idx']):
    chosen_idx = int(slim['chosen_idx'][step])

plot_mcts_tree_xy_limited_np(
    tree0,
    num_obstacles=len(slim.get('obstacles', [])),
    slim=slim,
    L=10,
    max_depth=6,
    top_k_per_node=5,
    chosen_child_idx=chosen_idx,
    title=f"Test",
)

<Axes: title={'center': 'Test'}>

In [10]:
dir(eval_run)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'eval_dir',
 'slim_traces',
 'summary',
 'trees_by_seed']

## Load summaries into a single DataFrame


In [11]:
runs = []
for d in eval_dirs:
    er = load_eval_dir(d)
    s = er.summary

    # infer run/ckpt name from .../<run>/eval/<ckpt>
    parts = er.eval_dir.parts
    run_name = parts[-3] if len(parts) >= 3 else str(er.eval_dir)
    ckpt_name = parts[-1]

    runs.append({
        'run': run_name,
        'ckpt': ckpt_name,
        'eval_dir': str(er.eval_dir),
        'checkpoint': s.get('checkpoint'),
        'step': s.get('step'),
        'env_variant': s.get('env_variant'),
        'num_seeds': s.get('num_seeds'),
        'max_steps': s.get('max_steps'),
        'tree_state_mode': s.get('tree_state_mode'),
        'return_mean': s.get('return_mean'),
        'return_std': s.get('return_std'),
        'success_rate': s.get('success_rate'),
        'crash_rate': s.get('crash_rate'),
    })

df = pd.DataFrame(runs)
if len(df):
    df = df.sort_values(['run', 'step']).reset_index(drop=True)
df


,run,ckpt,eval_dir,checkpoint,step,env_variant,num_seeds,max_steps,tree_state_mode,return_mean,return_std,success_rate,crash_rate
0,2d,ckpt_step_000020000,runs\base__seed42__9567da53\2d\eval\ckpt_step_...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,4.0,19.595919,0.04,0.0
1,2d,ckpt_step_000020000,runs\bootstrap_at_40k__seed42__976901ee\2d\eva...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,8.0,27.129320,0.08,0.0
2,2d,ckpt_step_000020000,runs\cpuct_0p2__seed42__4557fede\2d\eval\ckpt_...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,8.0,27.129320,0.08,0.0
3,2d,ckpt_step_000020000,runs\cpuct_20__seed42__49a8b2b7\2d\eval\ckpt_s...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,2.0,14.000000,0.02,0.0
4,2d,ckpt_step_000020000,runs\gamma_0p65__seed42__5737d1b2\2d\eval\ckpt...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,6.0,23.748684,0.06,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,2d,ckpt_step_000200000,runs\gamma_0p95__seed42__610886c9\2d\eval\ckpt...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,200000,2d,50,300,full,2.0,14.000000,0.02,0.0
98,2d,ckpt_step_000200000,runs\net_256x256__seed42__dafa96a9\2d\eval\ckp...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,200000,2d,50,300,full,2.0,14.000000,0.02,0.0
99,2d,ckpt_step_000200000,runs\pwk_1__seed42__1a86219a\2d\eval\ckpt_step...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,200000,2d,50,300,full,0.0,0.000000,0.00,0.0
100,2d,ckpt_step_000200000,runs\pwk_3__seed42__278f361c\2d\eval\ckpt_step...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,200000,2d,50,300,full,4.0,19.595919,0.04,0.0


### Best checkpoint selection helpers


In [14]:
def pick_best(
    df: pd.DataFrame,
    metric: str = "success_rate",
    *,
    group_by_run: bool = False,
    top_n: int = 1
):
    if df.empty:
        raise RuntimeError("No eval dirs found (df is empty).")

    if metric not in df.columns:
        raise KeyError(f"metric {metric} not in columns: {list(df.columns)}")

    if top_n < 1:
        raise ValueError("top_n must be >= 1")

    df_metric = df.copy()
    df_metric[metric] = df_metric[metric].astype(float)

    if group_by_run:
        # get best per run first
        best_per_run = df_metric.loc[
            df_metric.groupby("run")[metric].idxmax()
        ]
        best_per_run = best_per_run.sort_values(metric, ascending=False)
        return best_per_run.head(top_n)

    # global best N
    sorted_df = df_metric.sort_values(metric, ascending=False)
    return sorted_df.head(top_n)

best_overall = pick_best(df, metric='success_rate', group_by_run=False, top_n = 20)
best_overall


,run,ckpt,eval_dir,checkpoint,step,env_variant,num_seeds,max_steps,tree_state_mode,return_mean,return_std,success_rate,crash_rate
7,2d,ckpt_step_000020000,runs\pwk_1__seed42__1a86219a\2d\eval\ckpt_step...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,16.0,36.660606,0.16,0.0
40,2d,ckpt_step_000080000,runs\pwk_3__seed42__278f361c\2d\eval\ckpt_step...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,80000,2d,50,300,full,10.0,30.000000,0.10,0.0
41,2d,ckpt_step_000080000,runs\sims_400__seed42__7ef647c4\2d\eval\ckpt_s...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,80000,2d,50,300,full,10.0,30.000000,0.10,0.0
15,2d,ckpt_step_000040000,runs\cpuct_20__seed42__49a8b2b7\2d\eval\ckpt_s...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,40000,2d,50,300,full,10.0,30.000000,0.10,0.0
9,2d,ckpt_step_000020000,runs\sims_400__seed42__7ef647c4\2d\eval\ckpt_s...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,10.0,30.000000,0.10,0.0
10,2d,ckpt_step_000020000,runs\temp_0p2__seed42__b84f93a5\2d\eval\ckpt_s...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,20000,2d,50,300,full,14.0,40.049969,0.10,0.0
81,2d,ckpt_step_000160000,runs\temp_0p2__seed42__b84f93a5\2d\eval\ckpt_s...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,160000,2d,50,300,full,8.0,27.129320,0.08,0.0
30,2d,ckpt_step_000060000,runs\pwk_3__seed42__278f361c\2d\eval\ckpt_step...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,60000,2d,50,300,full,12.0,38.157570,0.08,0.0
53,2d,ckpt_step_000120000,runs\bootstrap_at_40k__seed42__976901ee\2d\eva...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,120000,2d,50,300,full,8.0,27.129320,0.08,0.0
57,2d,ckpt_step_000120000,runs\gamma_0p95__seed42__610886c9\2d\eval\ckpt...,C:\Users\timok\Desktop\rl_challenge\rl-comptet...,120000,2d,50,300,full,8.0,27.129320,0.08,0.0


## Inspect per-seed results for one eval directory


In [13]:
row = best_overall

eval_dir = Path(row['eval_dir'])
er = load_eval_dir(eval_dir)

print('Eval dir:', er.eval_dir)
print('Checkpoint:', er.summary.get('checkpoint'))
print('Step:', er.summary.get('step'))
print('Env:', er.summary.get('env_variant'))
print('Seeds:', len(er.slim_traces))
print('Available tree files:', len(er.trees_by_seed))

seed_df = pd.DataFrame(er.summary.get('per_seed', []))
seed_df = seed_df.sort_values('total_return', ascending=False).reset_index(drop=True)
seed_df


TypeError: expected str, bytes or os.PathLike object, not Series

### Plot a trajectory (slim trace)


In [ ]:
best_seed = int(seed_df.iloc[0]['seed']) if len(seed_df) else int(er.slim_traces[0]['seed'])
slim = next(s for s in er.slim_traces if int(s['seed']) == best_seed)

plot_seeker_trajectory_slim(
    slim,
    title=f"seed={best_seed} | return={slim['total_return']:.1f} | len={slim['ep_len']}",
    L=10,
)


### Plot an MCTS tree at a specific step

This uses `trees_seed_<seed>.npz` if present.


In [ ]:
seed = best_seed
step = 0

npz_path = er.trees_by_seed.get(int(seed))
if npz_path is None:
    raise FileNotFoundError(f"No trees file for seed={seed} in {er.eval_dir}")

packed = load_packed_trees(npz_path)
tree0 = slice_step_tree(packed, step)

chosen_idx = None
if 'chosen_idx' in slim and step < len(slim['chosen_idx']):
    chosen_idx = int(slim['chosen_idx'][step])

plot_mcts_tree_xy_limited_np(
    tree0,
    num_obstacles=len(slim.get('obstacles', [])),
    slim=slim,
    L=10,
    max_depth=6,
    top_k_per_node=5,
    chosen_child_idx=chosen_idx,
    title=f"seed={seed} | step={step}",
)


### Plot a debug-style step view (trajectory up to k + tree at k)


In [ ]:
k = 3

tree_k = slice_step_tree(packed, k)
plot_eval_step(
    slim,
    tree_k,
    k,
    num_obstacles=len(slim.get('obstacles', [])),
    L=10,
    max_depth=6,
    top_k_per_node=5,
    title=f"seed={seed} | step={k}",
)


## Interactive explorer

If you have `ipywidgets` installed, you can interactively browse seeds and steps.


In [ ]:
try:
    from ipywidgets import interact, IntSlider, Dropdown
except Exception as e:
    raise RuntimeError('ipywidgets not installed in this environment') from e

seed_options = sorted([int(s['seed']) for s in er.slim_traces])

@interact(
    seed=Dropdown(options=seed_options, value=seed_options[0], description='seed'),
    k=IntSlider(min=0, max=50, step=1, value=0, description='step'),
)
def _view(seed, k):
    slim = next(s for s in er.slim_traces if int(s['seed']) == int(seed))
    npz_path = er.trees_by_seed.get(int(seed))
    if npz_path is None:
        print(f"No trees file for seed={seed}")
        plot_seeker_trajectory_slim(slim, title=f"seed={seed}", L=10)
        return

    packed = load_packed_trees(npz_path)
    # trees are stored per action step (0..ep_len-1); agent_pos has length ep_len+1
    max_k = int(slim['ep_len']) - 1
    k = int(max(0, min(k, max_k)))

    tree_k = slice_step_tree(packed, k)
    plot_eval_step(
        slim,
        tree_k,
        k,
        num_obstacles=len(slim.get('obstacles', [])),
        L=10,
        max_depth=6,
        top_k_per_node=5,
        title=f"seed={seed} | step={k}",
    )
